# Standard NicheNet Workflow: From GRN to ccflowR Visualization

## Biological Question

Here we ask: **which sender-cell ligands are associated with basal epithelial gene-expression changes between contralateral (C) and normal adjacent (N) tissue?** This is a condition-comparison problem: the receiver cell type is fixed to basal epithelium, and the contrast is C vs N within those basal cells.

This tutorial is the main single-contrast workflow. It links four layers of interpretation:

1. **Differential receiver program**: genes that distinguish C from N basal cells.
2. **Ligand activity**: ligands whose predicted targets recover that receiver program.
3. **Ligand-target mechanisms**: target genes that connect top ligands to the receiver response.
4. **ccflowR Sankey view**: sender -> ligand -> receptor -> target -> receiver paths that make the communication hypothesis inspectable.

## Learning Objectives

By the end of this tutorial, you will be able to:

- Run the complete ContextNet -> NicheNet pipeline with condition comparison mode
- Interpret ligand activity scores and ligand-target regulatory potential
- Inspect `ContextGRN`, `ContextNetwork`, and `ContextResult` objects
- Extract matrices and tables for downstream analyses
- Render cell-cell communication as a Sankey diagram with ccflowR
- Save and reload ContextNet objects for reproducibility

## Overview

We study **signaling to basal epithelial cells** comparing contralateral (C) vs normal adjacent (N) tissue. This demonstrates **condition comparison mode** (`de_subset_receiver = TRUE`), which identifies ligands whose targets are differentially expressed between conditions within the receiver cell type.

This contrasts with Tutorial 01, which uses **cell type comparison** (`de_subset_receiver = FALSE`) to find basal-specific genes relative to other epithelial layers.

> **When to use this workflow:**
> - You have two conditions, such as disease/control or treated/untreated.
> - You want an interpretable single receiver-cell analysis.
> - You do not have biological replicates, or you intentionally want the simpler Wilcoxon-based workflow.
>
> **Use [03-multicondition-workflow](03-multicondition-workflow.ipynb) instead if:**
> - You have biological replicates and want pseudobulk, replicate-aware differential communication.

## Setup

In [ ]:
suppressPackageStartupMessages({
    library(ContextNet)
    library(Seurat)
    library(nichenetr)
    library(dplyr)
    library(ggplot2)
})

theme_set(theme_minimal(base_family = "sans"))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                      USER CONFIGURATION                      ║
# ║  Paths are relative to the ContextNet project root.          ║
# ╚══════════════════════════════════════════════════════════════╝

PROJECT_DIR <- normalizePath(getwd(), mustWork = TRUE)
if (basename(PROJECT_DIR) == "notebooks" &&
    file.exists(file.path(PROJECT_DIR, "..", "DESCRIPTION"))) {
    PROJECT_DIR <- normalizePath(file.path(PROJECT_DIR, ".."), mustWork = TRUE)
}
if (!file.exists(file.path(PROJECT_DIR, "DESCRIPTION"))) {
    stop("Run this notebook from the ContextNet repository root or from the notebooks/ directory.")
}
setwd(PROJECT_DIR)
cat("Working directory:", getwd(), "\n")

DATA_PATH       <- "data"
GRN_FILE        <- file.path(DATA_PATH, "gene_x_tf.csv")
SEURAT_FILE     <- file.path(DATA_PATH, "seurat_obj.rds")
NICHENET_DB     <- DATA_PATH

# Repo-local cache for the reusable ContextNetwork (a derived object; data/ is
# gitignored). Tutorial 03 reuses this if present, skipping re-integration.
CACHE_DIR       <- file.path(DATA_PATH, "cache")
NETWORK_CACHE   <- file.path(CACHE_DIR, "contextnet_network.rds")

# Analysis parameters: condition comparison for basal cells
SENDER          <- NULL           # Auto-detect: all cell types with >500 cells
RECEIVER        <- "basal"
GROUP_COL       <- "Fine_Cell_Types"
IDENT_COL       <- "category"     # Condition column (C vs N)
IDENT_1         <- "C"            # Contralateral (condition of interest)
IDENT_2         <- "N"            # Normal adjacent (reference)
MIN_CELLS       <- 500            # Minimum cells for sender cell types
TOP_N_LIGANDS   <- 30

stopifnot(file.exists(GRN_FILE), file.exists(SEURAT_FILE))


## Step 1: Import GRN

The context-specific GRN is a gene × TF matrix (CSV) where rows are target genes, columns are transcription factors, and values are regulatory weights inferred from matched scRNA-seq + scATAC-seq data.

In [ ]:
grn <- import_grn(GRN_FILE)
grn

In [ ]:
# Explore the GRN
cat("TFs:", length(tfs(grn)), "\n")
cat("Targets:", length(targets(grn)), "\n")
cat("Edges:", length(grn), "\n")
head(edges(grn))

## Step 2: Integrate with NicheNet

This replaces NicheNet's generic gene regulatory network with your context-specific GRN while preserving NicheNet's curated ligand-receptor interactions and signaling pathways. The result is a context-specific ligand-target matrix computed via Personalized PageRank.

Three integration modes are available:
- **`replace`** (default): Completely replace NicheNet's GRN — most context-specific
- **`augment`**: Add your GRN edges to NicheNet's existing network
- **`weighted`**: Weighted combination of both networks

> **Bring your own workflow:** The ligand-target matrix produced here is a standard R matrix. You can extract it with `ligand_target_matrix(network)` and plug it directly into any [nichenetr workflow](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_steps.md) in place of the default matrix. Steps 3-7 below use ContextNet's convenience wrappers, but they are optional.

### What the `ContextNetwork` Represents

`integrate_networks()` returns a `ContextNetwork`. This is the central reusable object in ContextNet: it stores a context-specific ligand-target matrix generated by propagating NicheNet ligand-receptor/signaling information through your GRN. A column is a ligand, a row is a target gene, and each entry is the regulatory potential connecting that ligand to that target in this biological context.

Because this object is independent of one sender/receiver contrast, you can save it once and reuse it across many downstream questions.

In [ ]:
# ~10 minutes — constructs context-specific ligand-target matrix
network <- integrate_networks(
    grn,
    nichenet_database = NICHENET_DB,
    mode = "replace"
)
network

In [ ]:
# Cache the network for reuse (e.g. Tutorial 03) so the ~10 min integration is
# not repeated. The cache is a repo-local derived object under data/, not /tmp.
if (!dir.exists(CACHE_DIR)) dir.create(CACHE_DIR, recursive = TRUE)
saveRDS(network, NETWORK_CACHE)
cat("Cached ContextNetwork to:", NETWORK_CACHE, "\n")
integration_params(network)

## Step 3: Load Expression Data

We compare contralateral (C) vs normal adjacent (N) tissue, focusing on signaling to basal epithelial cells. Basal cells are the proliferative progenitor layer of stratified squamous epithelium; condition-specific gene expression changes in basal cells may reflect early responses to microenvironmental signals from fibroblasts, immune cells, and endothelium.

In [ ]:
seurat_obj <- readRDS(SEURAT_FILE)
seurat_cn <- subset(seurat_obj, subset = category %in% c(IDENT_1, IDENT_2))
cat("Cells:", ncol(seurat_cn), "\n")
cat("Conditions:", paste(unique(seurat_cn$category), collapse = ", "), "\n")

In [ ]:
celltype_counts <- sort(table(seurat_cn[[GROUP_COL]][, 1]), decreasing = TRUE)
celltype_counts

In [ ]:
# Auto-detect senders if not specified
if (is.null(SENDER)) {
    # Abundance-based, so the auto-set can include the receiver cell type
    # itself as a candidate sender (autocrine); curate SENDER for a
    # focused hypothesis.
    SENDER <- names(celltype_counts)[celltype_counts > MIN_CELLS]
}

cat("Senders:", paste(SENDER, collapse = ", "), "\n")
cat("Receiver:", RECEIVER, "\n")
cat("Comparison:", IDENT_1, "vs", IDENT_2, "\n")

### Analysis Design Check

This workflow asks whether condition-associated basal-cell programs can be explained by ligands from the surrounding cell types. Before scoring ligands, verify that the selected sender and receiver populations are represented in both conditions and that the context-specific network overlaps the expression object.


In [ ]:
design_counts <- as.data.frame(table(
    cell_type = seurat_cn[[GROUP_COL]][, 1],
    condition = seurat_cn[[IDENT_COL]][, 1]
))
design_counts <- design_counts[design_counts$Freq > 0, ]
head(design_counts[order(design_counts$cell_type, design_counts$condition), ], 20)

expr_genes <- rownames(seurat_cn)
cat("GRN targets measured:", length(intersect(targets(grn), expr_genes)), "/", length(targets(grn)), "\n")
cat("Network targets measured:", length(intersect(targets(network), expr_genes)), "/", length(targets(network)), "\n")
cat("Network ligands measured:", length(intersect(ligands(network), expr_genes)), "/", length(ligands(network)), "\n")


## Step 4: Predict Ligand Activity

We use the default **condition comparison mode** (`de_subset_receiver = TRUE`). This subsets to basal cells first, then computes DE genes between contralateral (C) and normal adjacent (N) conditions. The question: **which ligands from the microenvironment drive condition-specific gene expression changes in basal cells?**

In [ ]:
# ~2 minutes — condition comparison: DE within basal cells between C and N
result <- predict_ligand_activity_nichenet(
    object = seurat_cn,
    network = network,
    sender = SENDER,
    receiver = RECEIVER,
    group_col = GROUP_COL,
    ident_col = IDENT_COL,
    ident_1 = IDENT_1,
    ident_2 = IDENT_2,
    nichenet_database = NICHENET_DB,
    top_n_ligands = TOP_N_LIGANDS
)

### Receiver Program

ContextNet first defines the basal-cell C-vs-N gene program, then asks which ligands recover that program through the context-specific ligand-target matrix. Inspecting the receiver program is essential: the ligand ranking is only meaningful relative to these genes.


In [ ]:
result_params <- parameters(result)
geneset_oi <- result_params$geneset_oi
cat("Receiver genes used for ligand activity:", length(geneset_oi), "\n")
cat("First receiver genes:", paste(head(geneset_oi, 25), collapse = ", "), "\n")
cat("Genes in ContextNetwork:", length(intersect(geneset_oi, rownames(ligand_target_matrix(network)))), "\n")


## Step 5: Explore Results

`result` is a `ContextResult`. It does not mean "the communication network" in general; it is the answer to the specific question encoded in the parameters: the selected senders, basal receiver cells, the C vs N contrast, and the filtering thresholds.

The most important accessors are:

| Accessor | What to inspect | Interpretation |
|----------|-----------------|----------------|
| `ligand_activities(result)` | full ranked ligand table | candidate upstream signals ranked by target recovery |
| `top_ligands(result)` | selected ligand names | ligands carried forward to plots and mechanism checks |
| `top_targets(result)` | high-priority target genes | receiver genes driving the ligand ranking |
| `ligand_target_links(result)` | ligand-target pairs | mechanistic hypotheses to validate |
| `parameters(result)` | analysis settings | exact biological question and thresholds |
| `as.data.frame(result)` | generic activity table | export to tidy workflows |

In [ ]:
top_ligands(result)

In [ ]:
head(ligand_activities(result), 10)

In [ ]:
head(top_targets(result), 20)

In [ ]:
head(ligand_target_links(result))

In [ ]:
parameters(result)

In [ ]:
# Extract ordinary R objects for downstream analysis
activities_tbl <- ligand_activities(result)
links_tbl <- ligand_target_links(result)
ltm <- ligand_target_matrix(network)
analysis_params <- parameters(result)

cat("Activity table:", nrow(activities_tbl), "ligands\n")
cat("Ligand-target links:", nrow(links_tbl), "rows\n")
cat("Ligand-target matrix:", nrow(ltm), "targets x", ncol(ltm), "ligands\n")

head(activities_tbl, 5)

These extracted tables are the recommended interface for interpretation and reporting. For example, use `activities_tbl` for a ranked ligand table, `links_tbl` to explain which target genes support each ligand, and `ltm` if you want to rerun custom nichenetr analyses outside the ContextNet wrapper.

## Step 6: Visualize

### Ligand Activity Bar Plot

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 8)

best_ligands <- top_ligands(result)
activities <- ligand_activities(result)

top_act <- activities[activities$test_ligand %in% best_ligands, ]
top_act$test_ligand <- factor(top_act$test_ligand, levels = rev(best_ligands))

ggplot(top_act, aes(x = aupr_corrected, y = test_ligand)) +
    geom_col(fill = "#2C7BB6") +
    labs(x = "Corrected ligand activity (AUPR_c)", y = NULL,
         title = "Top 30 Predicted Ligands (Basal Cells, C vs N)") +
    theme_minimal() +
    theme(plot.title = element_text(hjust = 0.5, face = "bold"))

**Interpretation**: The corrected AUPR (`AUPR_c`) is the area under the precision-recall curve for recovering the condition-specific basal DE genes, corrected against a null distribution of random ligands. Higher `AUPR_c` indicates that the ligand's predicted targets better overlap the genes changing between contralateral and normal adjacent tissue — a stronger candidate upstream regulator (a hypothesis, not proof).

### Ligand Activity Heatmap (nichenetr)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10)

activity_scores <- activities[activities$test_ligand %in% best_ligands, ]
activity_matrix <- as.matrix(activity_scores[, "aupr_corrected", drop = FALSE])
rownames(activity_matrix) <- activity_scores$test_ligand
colnames(activity_matrix) <- "AUPR_c"
activity_matrix <- activity_matrix[rev(best_ligands), , drop = FALSE]  # rev() so the highest score is the TOP row

nichenetr::make_heatmap_ggplot(
    activity_matrix,
    y_name = "Predicted Ligand",
    x_name = "",
    legend_title = "AUPR_c",
    color = "darkorange"
) + ggtitle("Ligand Activity Scores")

### Ligand-Target Heatmap

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 10)

best_targets <- top_targets(result, n = 50)
ltm <- ligand_target_matrix(network)

target_rows <- intersect(best_targets, rownames(ltm))
ligand_cols <- intersect(best_ligands, colnames(ltm))
vis_matrix <- ltm[target_rows, ligand_cols, drop = FALSE]

nonzero <- vis_matrix[vis_matrix > 0]
if (length(nonzero) > 0) {
    vis_matrix[vis_matrix < quantile(nonzero, 0.33)] <- 0
}
vis_matrix <- vis_matrix[rowSums(vis_matrix) > 0, , drop = FALSE]

print(nichenetr::make_heatmap_ggplot(
    vis_matrix,
    y_name = "Target Gene",
    x_name = "Predicted Ligand",
    legend_title = "Regulatory\nPotential",
    color = "purple"
))


**Interpretation**: This heatmap shows the regulatory potential between top ligands (columns) and their predicted target genes (rows) from the context-specific ligand-target matrix. Entries below the 33rd percentile are set to zero for visual clarity. Ligands with many non-zero entries regulate a broad set of targets; sparse columns indicate more specific regulatory effects.

### Sankey Visualization with ccflowR

The bar plot identifies candidate ligands, and the ligand-target heatmap identifies receiver genes supporting them. ccflowR answers the path-level question: **which sender -> ligand -> receptor -> target -> basal receiver routes explain the strongest C-vs-N basal-cell communication hypotheses?**

`prepare_for_ccflowr()` converts the `ContextResult` into path rows with sender, ligand, receptor, receiver, and score columns. `combine_sankey_data()` rescales and thresholds those rows so the graph emphasizes the most interpretable paths.

Interpretation rules:

- Wider flows are higher-priority paths after ContextNet scoring and global scaling.
- The sender node means the ligand is plausible from that cell population, not that contact has been proven.
- The receptor and target portions of the path are the mechanistic hypotheses to inspect in expression, perturbation, or spatial data.
- Use `threshold_quantile` as a visualization control: higher values show fewer, stronger paths; lower values show more hypotheses.


In [ ]:
# Build path-level Sankey data, then scale + threshold across receivers.
sankey_raw <- prepare_for_ccflowr(
    result,
    expression_data = seurat_cn,
    network = network,
    group_col = GROUP_COL,
    top_n_ligands = 50
)
sankey_data <- combine_sankey_data(sankey_raw, threshold_quantile = 0.95)

cat("Raw ccflowR path rows:", nrow(sankey_raw), "\n")
cat("Rows after thresholding:", nrow(sankey_data), "\n")
head(sankey_data)

In [ ]:
if (requireNamespace("ccflowR", quietly = TRUE)) {
    cell_types_in_plot <- unique(c(sankey_data$sender, sankey_data$receiver))
    palette <- grDevices::hcl.colors(length(cell_types_in_plot), palette = "Dark 3")
    cell_colors <- stats::setNames(palette, cell_types_in_plot)
    ccflowR::sankey_flow(sankey_data, cell_colors)
} else {
    message("ccflowR is not installed. Install it to render the Sankey graph: remotes::install_github('settylab/ccflowR')")
}


## Step 7: Export

### Save and Load Results

In [ ]:
tmp_dir <- tempdir()

# Save
save_contextnet(result, file.path(tmp_dir, "analysis.cnet"))

# Peek at saved file without loading
peek_contextnet(file.path(tmp_dir, "analysis.cnet"))

In [ ]:
# Reload later
result_loaded <- load_contextnet(file.path(tmp_dir, "analysis.cnet"))
top_ligands(result_loaded)

### Export Tables

In [ ]:
write.csv(ligand_activities(result), file.path(tmp_dir, "ligand_activities.csv"), row.names = FALSE)
write.csv(ligand_target_links(result), file.path(tmp_dir, "ligand_target_links.csv"), row.names = FALSE)

cat("Exported to:", tmp_dir, "\n")
list.files(tmp_dir, pattern = "\\.(cnet|csv)$")

In [ ]:
sessionInfo()

## Key Parameters

| Parameter | Required | Description |
|-----------|----------|-------------|
| `sender` | Yes | Sender cell type(s) expressing ligands |
| `receiver` | Yes | Receiver cell type |
| `ident_col` | Yes* | Metadata column for DE grouping |
| `ident_1` | Yes* | Value(s) for group 1 (upregulated genes); can be a vector |
| `ident_2` | Yes* | Value(s) for group 2 (reference); can be a vector |
| `de_subset_receiver` | No | `TRUE` (default): DE within receiver cells. `FALSE`: DE across all cells |
| `group_col` | No | Metadata column with cell type labels (default: `"cell_type"`) |
| `geneset_oi` | No | Custom gene set; auto-computed from DE if NULL |
| `top_n_ligands` | No | Number of top ligands (default: 30) |
| `nichenet_database` | No | Path to NicheNet data files |

*Required unless `geneset_oi` is provided directly.

## Using the Context-Specific Matrix with nichenetr Directly

ContextNet's reusable output from the integration step is the **context-specific ligand-target matrix**. If you want more control over the analysis, extract the matrix and use it in nichenetr's own functions:

```r
ltm <- ligand_target_matrix(network)

ligand_activities <- nichenetr::predict_ligand_activities(
    geneset = geneset_oi,
    background_expressed_genes = background_genes,
    ligand_target_matrix = ltm,
    potential_ligands = potential_ligands
)
```

This is useful when you already have custom DE results, want to test a curated gene set, need additional filtering, or want to compare the ContextNet matrix against the original NicheNet matrix.

For complete examples, see the nichenetr tutorials:
- [Step-by-step analysis](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_steps.md) - full walkthrough of NicheNet with Seurat
- [Seurat wrapper](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_wrapper.md) - simplified one-function interface
- [Circos visualization](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_wrapper_circos.md) - circos plots for ligand-receptor-target links
- [Ligand activity on custom gene set](https://github.com/saeyslab/nichenetr/blob/master/vignettes/ligand_activity_geneset.md) - use your own gene set